# Episode 02: Full Fine-Tuning (FFT) vs. PEFT & Memory Math

> **Getting Comfortable Series — Track 2: LLM Fine-Tuning, PEFT & Alignment**  
> *"Don't simplify the topic. Simplify the path to understanding it."*

$$\textbf{Understand} \longrightarrow \text{Build} \longrightarrow \text{Break} \longrightarrow \text{Introduce Tools} \longrightarrow \text{Apply} \longrightarrow \text{Revisit}$$

---

### What You Will Master in this Notebook
A common point of confusion for engineers entering LLM fine-tuning is the alphabet soup of terminology:
> *"Are we doing SFT or LoRA? Is FFT an alternative to SFT?"*

In this notebook, we eliminate this confusion forever:
1. **The 2D Orthogonal Mental Model:** Why Training Objective (What we teach) and Parameter Strategy (Which weights change) are completely separate axes.
2. **What FFT (Full Fine-Tuning) Actually Does:** PyTorch mechanics of `requires_grad = True` on 100% of parameters.
3. **The 16-Bytes-per-Parameter Memory Formula:** Why an 8-Billion parameter model ($pprox 16\text{ GB}$ file) requires **$80\text{--}100\text{ GB}$ of VRAM** for FFT.
4. **Adam Optimizer Anatomy:** Visualizing the first moment ($m$) and second moment ($v$) buffers that eat your GPU memory.
5. **Interactive Hardware Memory Estimator:** Build a Python calculator to compute exact VRAM needs for FFT, LoRA, and QLoRA across any model size.


## 0. Setup & Environment

Let's import PyTorch to inspect tensor memory allocations directly.


In [ ]:
import torch
import torch.nn as nn

print("PyTorch version:", torch.__version__)
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print("Active Device:", device)


---

## 1. Demystifying the Acronyms: The 2D Orthogonal Grid

The #1 conceptual mistake is treating **SFT** and **LoRA** as opposing options. They are orthogonal choices on two different axes:

$$\begin{array}{c|c|c|c}
& \textbf{PEFT / LoRA (Adapters)} & \textbf{QLoRA (4-bit Base + Adapters)} & \textbf{FFT (Full Fine-Tuning)} \\
\hline
\textbf{SFT (Instruction Tuning)} & \text{LoRA SFT (Standard Enterprise)} & \text{QLoRA SFT (Single-GPU Standard)} & \text{Full SFT (Foundation Labs)} \\
\hline
\textbf{DPO (Preference Alignment)} & \text{LoRA DPO} & \text{QLoRA DPO} & \text{Full DPO} \\
\hline
\textbf{GRPO / PPO (RL Alignment)} & \text{LoRA GRPO} & \text{QLoRA GRPO} & \text{Full GRPO} \\
\end{array}$$

* **Axis 1: The Objective / Data (WHAT you are teaching):**
  * **SFT (Supervised Fine-Tuning):** Prompt-response pairs $(x, y)$. Teaches conversational structure, task formatting, and tool calling.
  * **DPO (Direct Preference Optimization):** Paired responses $(x, y_{\text{chosen}}, y_{\text{rejected}})$. Teaches preferences and style without a separate reward model.
  * **GRPO / PPO:** Rule-based or learned rewards. Teaches multi-step reasoning and verification.

* **Axis 2: The Weight Strategy (WHICH parameters change):**
  * **FFT (Full Fine-Tuning):** $100\%$ of parameters are updated.
  * **PEFT (LoRA):** $99.8\%$ of parameters are frozen ($W_0$); only tiny low-rank matrices ($B \times A$) are updated.
  * **QLoRA:** Frozen base weights are compressed into 4-bit NormalFloat; adapters remain 16-bit.


In [ ]:
# Let's inspect this in PyTorch code!
toy_model = nn.Sequential(
    nn.Linear(10, 10),
    nn.Linear(10, 10),
    nn.Linear(10, 2)
)

total_params = sum(p.numel() for p in toy_model.parameters())
trainable_params_fft = sum(p.numel() for p in toy_model.parameters() if p.requires_grad)

print(f"Total model parameters: {total_params}")
print(f"Trainable parameters in FFT: {trainable_params_fft} (100% of weights updated!)")


---

## 2. Why Does Full Fine-Tuning (FFT) Explode GPU Memory?

If an 8-Billion parameter model takes $16\text{ GB}$ on disk (in 16-bit float), why can't we train it on an RTX 3090 / 4090 with $24\text{ GB}$ of VRAM?

Because during training, **the model weights are only a small fraction of the total memory!**

Let's break down the memory required per parameter during FFT with the standard **AdamW** optimizer:

$$\textbf{Total FFT Memory Per Parameter} = \mathbf{16 \text{ to } 20 \text{ Bytes}}$$

Let's see where every single byte goes:

| Component | Precision | Bytes per Param | For 8B Model |
|---|:---:|:---:|:---:|
| **1. Model Weights** | FP16 / BF16 | $2\text{ bytes}$ | $16\text{ GB}$ |
| **2. Gradients ($\nabla_W \mathcal{L}$)** | FP16 / BF16 | $2\text{ bytes}$ | $16\text{ GB}$ |
| **3. Optimizer First Moment ($m$)** | FP32 | $4\text{ bytes}$ | $32\text{ GB}$ |
| **4. Optimizer Second Moment ($v$)** | FP32 | $4\text{ bytes}$ | $32\text{ GB}$ |
| **5. Master Weights (for stable Adam)** | FP32 | $4\text{ bytes}$ | $32\text{ GB}$ |
| **Static Training Footprint (without activations)** | | **$16\text{ bytes}$** | **$128\text{ GB}$!** |


In [ ]:
def breakdown_fft_memory(num_params_billions):
    params = num_params_billions * 1e9
    
    weights_gb   = (params * 2) / 1e9 # FP16
    grads_gb     = (params * 2) / 1e9 # FP16
    adam_m_gb    = (params * 4) / 1e9 # FP32 first moment
    adam_v_gb    = (params * 4) / 1e9 # FP32 second moment
    master_w_gb  = (params * 4) / 1e9 # FP32 master weights (in mixed-precision)
    
    static_total = weights_gb + grads_gb + adam_m_gb + adam_v_gb + master_w_gb
    
    print(f"=== Memory Breakdown for a {num_params_billions}B Model (Full Fine-Tuning) ===")
    print(f"1. Model Weights (FP16):           {weights_gb:>6.1f} GB")
    print(f"2. Gradients (FP16):               {grads_gb:>6.1f} GB")
    print(f"3. Adam First Moment (m, FP32):    {adam_m_gb:>6.1f} GB")
    print(f"4. Adam Second Moment (v, FP32):   {adam_v_gb:>6.1f} GB")
    print(f"5. FP32 Master Weights:            {master_w_gb:>6.1f} GB")
    print("-" * 50)
    print(f"STATIC VRAM REQUIRED (No activations): {static_total:>6.1f} GB")

breakdown_fft_memory(num_params_billions=8.0)


---

## 3. How PEFT (LoRA) and QLoRA Slash this Equation

Now look at what happens when we switch to **LoRA**:
1. Base model weights $W_0$ are **FROZEN** (`requires_grad = False`).
   - No gradients computed for $W_0$ ($0\text{ GB}$ gradient memory).
   - No Adam optimizer states tracked for $W_0$ ($0\text{ GB}$ Adam memory).
2. Gradients and Adam states are **only stored for the adapter matrices $B$ and $A$** (which is $<0.2\%$ of the model!).

And in **QLoRA**:
- The frozen base weights are stored in **4-bit NormalFloat** ($\sim 0.5\text{ bytes/parameter}$).
- An 8B base model shrinks from $16\text{ GB}$ to just **$\sim 5\text{ GB}$**!


In [ ]:
def compare_strategies(num_params_billions):
    params = num_params_billions * 1e9
    
    # 1. Full Fine-Tuning (~16 bytes per param)
    fft_gb = (params * 16) / 1e9
    
    # 2. LoRA (FP16 Base ~2 bytes + small adapters ~0.1 bytes)
    lora_gb = (params * 2) / 1e9 + 1.5 # base + adapter buffer
    
    # 3. QLoRA (4-bit Base ~0.55 bytes + adapters ~0.1 bytes)
    qlora_gb = (params * 0.55) / 1e9 + 1.5
    
    print(f"Hardware Footprint Comparison for a {num_params_billions}B Model:")
    print(f"  - Full Fine-Tuning (FFT): ~{fft_gb:.1f} GB VRAM  --> Requires 2x A100 (80GB) GPUs")
    print(f"  - LoRA (16-bit Base):     ~{lora_gb:.1f} GB VRAM  --> Fits on 1x RTX 3090/4090 (24GB)")
    print(f"  - QLoRA (4-bit Base):    ~{qlora_gb:.1f} GB VRAM  --> Fits on free Google Colab (T4 16GB)!")

compare_strategies(num_params_billions=8.0)


---

## 4. When is Full Fine-Tuning (FFT) Actually Needed?

If LoRA is so efficient, why does anyone ever do Full Fine-Tuning?

| Scenario | Recommended Strategy | Why? |
|---|:---:|---|
| **Adding a brand-new human language** (e.g., teaching English Llama to speak Tamil or Icelandic) | **FFT** | Requires reshaping the foundational semantic embedding manifold across all layers; low rank is too constrained. |
| **Major domain shifts from scratch** (e.g. raw protein folding or binary assembly decompilation) | **FFT** | Base representations are fundamentally missing the underlying token relationships. |
| **Format enforcement, strict JSON, SQL generation** | **LoRA / QLoRA** | The model already understands English and SQL syntax; it only needs to learn the output *schema constraint* (low intrinsic rank). |
| **Tone adaptation, brand voice, persona alignment** | **LoRA / QLoRA** | Modifying style requires altering probability distributions over synonyms, an inherently low-rank change. |
| **Agentic tool dispatch & SRE alert routing** | **LoRA / QLoRA** | Learning internal company schemas and routing rules does not require changing core language understanding. |


In [ ]:
# The Decision Helper Function
def should_i_fft(task_description: str, has_massive_data: bool, has_gpu_cluster: bool):
    print(f"Evaluating task: '{task_description}'")
    if not has_gpu_cluster:
        return "Verdict: Use QLoRA / LoRA. FFT is hardware-prohibitive without a multi-GPU cluster."
    if "new language" in task_description.lower() and has_massive_data:
        return "Verdict: FFT recommended. Foundational vocabulary and grammar expansion benefits from full weights."
    return "Verdict: Use LoRA / QLoRA. Adapter tuning achieves 99%+ of FFT performance at a fraction of the cost."

print(should_i_fft("Incident routing to 15 JSON teams", has_massive_data=False, has_gpu_cluster=False))
print(should_i_fft("Teaching Icelandic to Llama 3", has_massive_data=True, has_gpu_cluster=True))


---

## 🎯 Episode 02 Comfort Recap

1. **SFT and LoRA are not opposites:** SFT is the *objective* (instruction pairs); LoRA is the *parameter strategy* (low-rank adapters).
2. **FFT updates 100% of weights:** Every single weight tensor computes gradients and maintains Adam momentum buffers.
3. **The 16-Bytes-per-Param Rule:** In FFT, an 8B model requires $\sim 128\text{ GB}$ of static VRAM before adding activations.
4. **LoRA slashes memory** because frozen base weights have zero gradient and zero optimizer memory overhead.
5. **Rule of thumb for production:** Use LoRA / QLoRA for $95\%$ of engineering tasks (formatting, routing, JSON, tone). Reserve FFT strictly for massive domain shifts or new human languages.

---

### Up Next:
In **Episode 03: SFT Mechanics & Loss Math**, we look at the Supervised Fine-Tuning objective from first principles:
- The Autoregressive Cross-Entropy equation.
- Hand-coding token shifting (`logits[:-1]` vs `labels[1:]`).
- How SFT breaks the "autocomplete reflex" and turns an LLM into a structured assistant.
